# Image Segmentation with U-Net for Pothole Detection

This notebook demonstrates semantic image segmentation using a **U-Net** architecture with a
**ResNet-34** encoder backbone, trained on the
[Pothole Image Segmentation Dataset](https://www.kaggle.com/datasets/raunakkesharwani/pothole-image-segmentation-dataset).

**Pipeline overview:**
1. Load images and binary masks from the dataset
2. Apply data augmentation with Albumentations
3. Train a U-Net model using a combined Dice + BCE loss
4. Evaluate with Intersection-over-Union (IoU) on the validation set
5. Visualize predictions against ground truth

**Key libraries:** PyTorch, segmentation-models-pytorch, Albumentations, OpenCV

In [ ]:
# ============================================================
# Cell 1 - Install & Import
# ============================================================

!pip install -q segmentation-models-pytorch albumentations

import os
import glob
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Cell 2 - Configuration
# ============================================================

IMG_SIZE    = 256        # Resize all images to IMG_SIZE x IMG_SIZE
BATCH_SIZE  = 8
NUM_EPOCHS  = 15
LR          = 1e-3
ENCODER     = "resnet34" # Encoder backbone for U-Net
WEIGHTS     = "imagenet" # Pretrained encoder weights
NUM_CLASSES = 1          # Binary segmentation (pothole vs background)

# Dataset paths (Kaggle default mount point)
DATA_ROOT  = "/kaggle/input/pothole-image-segmentation-dataset"
IMAGE_DIR  = os.path.join(DATA_ROOT, "images")
MASK_DIR   = os.path.join(DATA_ROOT, "masks")

# Adjust paths if the dataset has a nested structure
if not os.path.isdir(IMAGE_DIR):
    # Try common alternative layouts
    for candidate in sorted(glob.glob(os.path.join(DATA_ROOT, "**", "images"), recursive=True)):
        IMAGE_DIR = candidate
        MASK_DIR  = candidate.replace("images", "masks")
        break

print(f"Image dir: {IMAGE_DIR}")
print(f"Mask dir:  {MASK_DIR}")

In [ ]:
# ============================================================
# Cell 3 - Custom Dataset
# ============================================================

class PotholeDataset(Dataset):
    """PyTorch Dataset for loading pothole images and their binary masks."""

    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image in RGB and mask in grayscale
        image = cv2.imread(self.image_paths[idx], cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)

        # Binarise the mask (threshold at 127)
        mask = (mask > 127).astype(np.float32)

        # Apply albumentations transforms
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]

        # Ensure mask has a channel dimension: (1, H, W)
        if isinstance(mask, torch.Tensor) and mask.ndim == 2:
            mask = mask.unsqueeze(0)
        elif isinstance(mask, np.ndarray) and mask.ndim == 2:
            mask = np.expand_dims(mask, axis=0)

        return image, mask

In [ ]:
# ============================================================
# Cell 4 - Data Loading & Augmentation
# ============================================================

# Collect and sort file paths
image_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.*")))
mask_paths  = sorted(glob.glob(os.path.join(MASK_DIR, "*.*")))

assert len(image_paths) == len(mask_paths), (
    f"Mismatch: {len(image_paths)} images vs {len(mask_paths)} masks"
)
print(f"Total samples: {len(image_paths)}")

# Train / validation split (80-20)
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    image_paths, mask_paths, test_size=0.2, random_state=SEED
)
print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")

# Augmentation pipelines
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Build datasets and loaders
train_dataset = PotholeDataset(train_imgs, train_masks, transform=train_transform)
val_dataset   = PotholeDataset(val_imgs,   val_masks,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ============================================================
# Cell 5 - Visualise Sample Images & Masks
# ============================================================

def show_samples(dataset, n=4):
    """Display n random image-mask pairs from the dataset (pre-augmentation)."""
    fig, axes = plt.subplots(n, 2, figsize=(8, 3 * n))
    indices = random.sample(range(len(dataset)), n)

    for row, idx in enumerate(indices):
        # Read raw image and mask for visualisation
        img  = cv2.imread(dataset.image_paths[idx], cv2.IMREAD_COLOR)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(dataset.mask_paths[idx], cv2.IMREAD_GRAYSCALE)

        axes[row, 0].imshow(img)
        axes[row, 0].set_title("Image")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask, cmap="gray")
        axes[row, 1].set_title("Ground Truth Mask")
        axes[row, 1].axis("off")

    plt.tight_layout()
    plt.show()

show_samples(train_dataset, n=4)

In [ ]:
# ============================================================
# Cell 6 - Model Definition
# ============================================================

model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=WEIGHTS,
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None,  # Raw logits; we apply sigmoid in post-processing
)
model = model.to(device)

# Quick sanity check with a dummy input
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
with torch.no_grad():
    out = model(dummy)
print(f"Model output shape: {out.shape}  (expected [1, 1, {IMG_SIZE}, {IMG_SIZE}])")

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total_params:,} total | {trainable:,} trainable")

In [ ]:
# ============================================================
# Cell 7 - Training Loop
# ============================================================

# Loss: combination of Dice loss and BCE with logits
dice_loss_fn = DiceLoss(mode="binary", from_logits=True)
bce_loss_fn  = nn.BCEWithLogitsLoss()

def combined_loss(pred, target):
    return 0.5 * dice_loss_fn(pred, target) + 0.5 * bce_loss_fn(pred, target)

# IoU / Jaccard metric
def compute_iou(pred_logits, target, threshold=0.5):
    """Compute mean IoU for a batch of predictions."""
    preds = (torch.sigmoid(pred_logits) > threshold).float()
    intersection = (preds * target).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + 1e-7) / (union + 1e-7)
    return iou.mean().item()

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, verbose=True
)

# History tracking
history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}
best_val_iou = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Training phase ---
    model.train()
    running_loss, running_iou, n_batches = 0.0, 0.0, 0

    for images, masks in train_loader:
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_iou  += compute_iou(outputs, masks)
        n_batches    += 1

    train_loss = running_loss / n_batches
    train_iou  = running_iou  / n_batches

    # --- Validation phase ---
    model.eval()
    running_loss, running_iou, n_batches = 0.0, 0.0, 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks  = masks.to(device)

            outputs = model(images)
            loss = combined_loss(outputs, masks)

            running_loss += loss.item()
            running_iou  += compute_iou(outputs, masks)
            n_batches    += 1

    val_loss = running_loss / n_batches
    val_iou  = running_iou  / n_batches

    scheduler.step(val_loss)

    # Record history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_iou"].append(train_iou)
    history["val_iou"].append(val_iou)

    # Save best model
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), "best_unet_pothole.pth")

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}  IoU: {train_iou:.4f} | "
        f"Val Loss: {val_loss:.4f}  IoU: {val_iou:.4f}"
    )

print(f"\nTraining complete. Best validation IoU: {best_val_iou:.4f}")

In [ ]:
# ============================================================
# Cell 8 - Evaluation & Visualisation
# ============================================================

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)
ax1.plot(epochs_range, history["train_loss"], label="Train Loss")
ax1.plot(epochs_range, history["val_loss"],   label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curve")
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, history["train_iou"], label="Train IoU")
ax2.plot(epochs_range, history["val_iou"],   label="Val IoU")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("IoU")
ax2.set_title("IoU Curve")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# --- Load best weights and compute mean IoU on validation set ---
model.load_state_dict(torch.load("best_unet_pothole.pth", map_location=device))
model.eval()

total_iou, count = 0.0, 0
with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks  = masks.to(device)
        preds  = model(images)
        total_iou += compute_iou(preds, masks) * images.size(0)
        count += images.size(0)

mean_iou = total_iou / count
print(f"Mean IoU on validation set (best model): {mean_iou:.4f}")

# --- Side-by-side predictions vs ground truth ---
def show_predictions(model, dataset, n=6):
    """Display model predictions alongside ground truth masks."""
    model.eval()
    indices = random.sample(range(len(dataset)), n)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))

    for row, idx in enumerate(indices):
        image_tensor, mask_tensor = dataset[idx]

        # Predict
        with torch.no_grad():
            pred_logits = model(image_tensor.unsqueeze(0).to(device))
            pred_mask = (torch.sigmoid(pred_logits) > 0.5).cpu().squeeze().numpy()

        # De-normalise image for display
        img_np = image_tensor.permute(1, 2, 0).numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img_np = (img_np * std + mean).clip(0, 1)

        gt_mask = mask_tensor.squeeze().numpy()

        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title("Input Image")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(gt_mask, cmap="gray")
        axes[row, 1].set_title("Ground Truth")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred_mask, cmap="gray")
        axes[row, 2].set_title("Prediction")
        axes[row, 2].axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(model, val_dataset, n=6)

In [ ]:
# ============================================================
# Cell 9 - Save Model
# ============================================================

save_path = "unet_pothole_final.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "encoder": ENCODER,
    "img_size": IMG_SIZE,
    "num_classes": NUM_CLASSES,
    "best_val_iou": best_val_iou,
    "history": history,
}, save_path)

print(f"Model checkpoint saved to: {save_path}")
print(f"  Encoder:       {ENCODER}")
print(f"  Image size:    {IMG_SIZE}")
print(f"  Best val IoU:  {best_val_iou:.4f}")